# Downloading information from website Constitutional Court

The website of the Belgian Constitutional Court contains all judgements (on a different page per year) as well as assigned keywords. For instance, for 2023, all cases and the corresponding keywords can be found at https://www.const-court.be/nl/judgments?year=2023. Below we extract the cases and link them to their corresponding keywords.

## Setting up

In [1]:
import wget
import requests
from urllib.request import urlopen
from bs4 import BeautifulSoup
import pandas as pd

In [2]:
# show all outputs of cell, not merely of last line (i.e. default of Jupyter Notebook)
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [ ]:
# Assign folder to downlad cases into + hyperlink with webpage containing judgements and corresponding keywords
# download_folder = "C:\\Users\\niels\\Documents\\Documenten\\Data science\\MaStat\\Masterproef\\repository\\data\\CC_judgments"
download_folder = "../data"

In [4]:
# Determine temporal scope of years to extract, e.g. all cases from 2000 until 2023
scope_years = [year for year in range(2025,2026)]
print("The scope of years is: ", scope_years)

The scope of years is:  [2025]


## Extracting relevant data

Relevant sources for assessing structure of html + extractig elements: 
* inspect source code of website using right mouse click 'inspect' or 'show source code'
* https://beautiful-soup.readthedocs.io/en/latest/#modifying-the-tree
* https://www.dataquest.io/blog/web-scraping-beautifulsoup/
* https://stackoverflow.com/questions/30561260/python-change-accept-language-using-requests
* https://www.geeksforgeeks.org/find-the-text-of-the-given-tag-using-beautifulsoup/
* https://opensource.com/article/21/9/web-scraping-python-beautiful-soup

First, we define a function to obtain a relevant BeautifulSoup object for the webpage containing all cases of a particular year.

In [5]:
def get_BeautifulSoup (year, language = "nl"):
    """
    Function to obtain BeautifulSoup object of webpage of Constitutional Court
    that contains the judgments of the year you indicate
    """
    
    # Set cases_url to web page of year you want to extract cases of
    cases_url = "https://www.const-court.be/nl/judgments?year=" + year
    
    # Set headers to obtain language version NL 
    # see https://stackoverflow.com/questions/30561260/python-change-accept-language-using-requests + https://www.dataquest.io/blog/web-scraping-beautifulsoup/
    headers_language = {"Accept-Language": language} 
    
    # Obtain html code of CoC website for chosen language version
    r = requests.get(cases_url, headers=headers_language)
    
    # Create BeautifulSoup object
    bs = BeautifulSoup(r.text, 'html.parser')
    
    return bs

Then we create empty variables for all relevant information, to fill in during web scraping.

In [6]:
# create empty variables to store
cases_dates = []
cases_type_procedure = []
cases_keywords = []
cases_casenumber = []
cases_hyperlink = []
cases_controlled_norm = []
cases_outcome = []
cases_rolenumber = []

For every year in the scope, we then extract the relevant information.

In [7]:
%%time

for year in scope_years: 
    # Obtain BeautifulSoup object for each year in the range
    bs = get_BeautifulSoup(str(year))
                           
    # Obtain sections of html page that relate to each specific case
    all_cases = bs.find_all("div",  {"class": "judgement-card mx-auto my-4 v-card v-sheet theme--light"})

    # For each case, obtain the relevant information
    for case in all_cases:
        # Obtain all information that resides in top row of each judgement card (i.e. date and type of procedure)
        for information in case.find_all("div", {"class":  "top-infos"}):
            date, type_procedure = information.find_all("p")
            cases_dates.append(date.get_text())
            cases_type_procedure.append(type_procedure.get_text())

        # Obtain case number and link. This is stored in first 'h3' heading , witin 'a'.
        # attr. is used to extract the specific hyperlink itself as stored in 'href'
        for information in case.find_all("h3"): 
            # Obtain hyperlink
            hyperlink = information.a.attrs["href"]
            # Append hyperlink (+ add main website to hyperlink since this is always left out)
            cases_hyperlink.append("www.const-court.be" + hyperlink)

            # Obtain case number. There does not seem to be a specific tag for this.
            # But the case number is always preceded with an empty html comment: "<!-- -->" (https://htmlcheatsheet.com/).
            # Hence we use this to identify the case numbers within the string representation of information
            location_comment = str(information).find("<!-- -->")
            # select all text following this empty comment (this is the last element in the html string)
            casenumber = str(information)[location_comment:]
            # Strip resulting string of all trailing elements
            casenumber = casenumber.replace("<!-- -->\n", "").replace("</h3>", "").replace("\n", "")
            casenumber = casenumber.strip(" ")
            cases_casenumber.append(casenumber) # append actual case number        



        # Obtain controlled_norm, outcome, rolenumber which are in that order stored in 'span' 
        information = case.find_all("span")
        try:
            controlled_norm, outcome, rolenumber, keywords = information   
            # Append text of those elements (using '.get_text') to main lists, stripped if necessary
            cases_controlled_norm.append(controlled_norm.get_text())
            cases_outcome.append(outcome.get_text())
            # Strip rolenumbers, split into list if more than one, and append
            rolenumber = rolenumber.get_text().replace("\n", "").replace("Rolnummer: ", "").strip(" ")
            rolenumber = rolenumber.split(" - ")
            cases_rolenumber.append(rolenumber)
            # Append keywords   
            keywords = keywords.get_text().replace("\n", "").replace("Trefwoorden: ", "").strip(" ")
            # Split keywords into list if more than one
            keywords = keywords.split(" - ")
            cases_keywords.append(keywords)
        except:
            # If there is no information, append empty string
            cases_controlled_norm.append("")
            cases_outcome.append("")
            cases_rolenumber.append("")
            cases_keywords.append("")
            # print("No information available for this case")

CPU times: user 67.5 ms, sys: 17 ms, total: 84.5 ms
Wall time: 961 ms


This results in the following data set:

In [8]:
# # Assess results
# cases_hyperlink
# cases_casenumber
# cases_dates
# cases_type_procedure   
# cases_controlled_norm
# cases_keywords
# cases_outcome
# cases_rolenumber

In [9]:
# Merge all lists to dataframe (with dictionary as intermediary step)
data_dict = {
    'case number': cases_casenumber,
    'role number': cases_rolenumber,
    'date': cases_dates,
    'type procedure': cases_type_procedure,
    'controlled norm': cases_controlled_norm,
    'outcome': cases_outcome,
    'keywords': cases_keywords,
    'url': cases_hyperlink
          }

In [10]:
data = pd.DataFrame(data_dict)

# inspect result
data

,case number,role number,date,type procedure,controlled norm,outcome,keywords,url
0,61/2025,[8423],03/04/2025,Beroep tot vernietiging,Wetten en procedures inzake gedeeltelijke verb...,Verwerping van het beroep,"[Voorafgaande rechtspleging, Beroep tot vernie...",www.const-court.be/public/n/2025/2025-061n.pdf
1,60/2025,[8286],03/04/2025,Prejudiciële vraag,"Wetboek van economisch recht (artikel I.1, eer...",Onontvankelijkheid van de prejudiciële vraag,"[Economisch recht, Insolventie van onderneming...",www.const-court.be/public/n/2025/2025-060n.pdf
2,59/2025,[8285],03/04/2025,Beroep tot vernietiging,Ordonnantie van het Brusselse Hoofdstedelijke ...,Vernietiging,"[Fiscaal recht, Registratierechten, Brusselse ...",www.const-court.be/public/n/2025/2025-059n.pdf
3,58/2025,[8266],03/04/2025,Beroep tot vernietiging,Programmawet van 22 december 2023 (artikel 114),Verwerping van het beroep,"[Sociaal recht, Sociale zekerheid, Ziekte- en ...",www.const-court.be/public/n/2025/2025-058n.pdf
4,57/2025,"[8224, 8223]",03/04/2025,Prejudiciële vragen,Wet van 17 april 1878 « houdende de voorafgaan...,Geen schending (artikel 26 van de wet van 17 a...,"[Sociale zekerheid, Ziekte- en invaliditeitsve...",www.const-court.be/public/n/2025/2025-057n.pdf
...,...,...,...,...,...,...,...,...
56,5/2025,"[8157, 8156]",16/01/2025,Beroepen tot vernietiging,Decreet van het Vlaamse Gewest van 23 juni 202...,"- Vernietiging (artikel 5, eerste lid, 1°, van...","[Bestuursrecht, Grond- en pandenbeleid, Wonen ...",www.const-court.be/public/n/2025/2025-005n.pdf
57,4/2025,[8154],16/01/2025,Prejudiciële vraag,Wetboek van de inkomstenbelastingen 1992 (arti...,"Geen schending (artikel 269, § 2, eerste lid, ...","[Fiscaal recht, Inkomstenbelastingen, Personen...",www.const-court.be/public/n/2025/2025-004n.pdf
58,3/2025,[8148],16/01/2025,Beroep tot vernietiging,Ordonnantie van het Brusselse Hoofdstedelijke ...,- Vernietiging- Handhaving van de gevolgen van...,"[Ruimtelijke ordening, Brussels Hoofdstedelijk...",www.const-court.be/public/n/2025/2025-003n.pdf
59,2/2025,[8144],09/01/2025,Prejudiciële vraag,Brusselse Huisvestingscode (artikel 10),"Schending (artikel 10, § 3, van de Brusselse H...","[Huisvesting, Brussels Hoofdstedelijk Gewest, ...",www.const-court.be/public/n/2025/2025-002n.pdf


In [11]:
# Obtain differtent data types of dataframe (to be later on used when reading in data again)
data.dtypes

case number        object
role number        object
date               object
type procedure     object
controlled norm    object
outcome            object
keywords           object
url                object
dtype: object

After visual inspection of the data set, it appears there are some anomalies which can be corrected.
* First, for some cases a corrected version has been uploaded. This still shows in the variable "type procedure". For these cases, the string the date of correction and the string "Beschikking tot verbetering" are added. It appears this is the case for:
    * case number 114/2005 of 30/06/2005 with index 973
    * case number 102/2003 of 22/07/2003 with index 570
    
* Also for case number 82/2005 of 27/04/2005 with index 1005, the variable "outcome" contains an extra newline character. This should be stripped.
* For case number 101/2008 of 10/07/2008 with index 1537, the variable "case number" contains an extra carriage return. This should be stripped


In [12]:
# # Create boolean that indicates if a case contains "Beschikking tot verbetering" at variable "type procedure"
# # see "https://thecodingbot.com/check-if-a-column-contains-specific-string-in-a-pandas-dataframe/"
# beschikking_tot_verbetering_bool = data["type procedure"].str.contains("Beschikking tot verbetering")
# # Inspect for which cases such string is present
# data[beschikking_tot_verbetering_bool]
# # Data correction
# data.iloc[973]["type procedure"] = "Beroepen tot vernietiging"
# data.iloc[570]["type procedure"] = "Beroepen tot vernietiging"

# data.iloc[1005]["outcome"] = data.iloc[1005]["outcome"].strip("\n").strip(" ")
# # data.iloc[1005]["outcome"]

# data.iloc[1537]["case number"] = data.iloc[1537]["case number"].strip("\r")

In [13]:
# # Check result
# beschikking_tot_verbetering_bool = data["type procedure"].str.contains("Beschikking tot verbetering")
# data[beschikking_tot_verbetering_bool]

For some cases, no keywords are included in the dataset (only "-"). These are generally cases that are of no legal interest, e.g. when  the demanding party renounces the its claim, when the preliminary question does not require an answer ...  Since those cases are of no interest to assign keywords, they can be disregarded.

In [14]:
# First transform list of keywords to string to easier assess if the keyword varable only contains "-"  
keywords_str = [''.join(keywords_list) for keywords_list in data["keywords"]]
# Then create a boolean to select those cases
missing_values_keywords_bool = [keyword_str == "-" for keyword_str in keywords_str]
# Using that boolean, assess those cases with missing values
data[missing_values_keywords_bool]
# Determine amount of cases with missing values
len(data[missing_values_keywords_bool])

,case number,role number,date,type procedure,controlled norm,outcome,keywords,url


0

Then we actually delete those cases out of the data.

In [15]:
# see https://stackoverflow.com/questions/13851535/how-to-delete-rows-from-a-pandas-dataframe-based-on-a-conditional-expression
data = data.drop(data[missing_values_keywords_bool].index)
data

,case number,role number,date,type procedure,controlled norm,outcome,keywords,url
0,61/2025,[8423],03/04/2025,Beroep tot vernietiging,Wetten en procedures inzake gedeeltelijke verb...,Verwerping van het beroep,"[Voorafgaande rechtspleging, Beroep tot vernie...",www.const-court.be/public/n/2025/2025-061n.pdf
1,60/2025,[8286],03/04/2025,Prejudiciële vraag,"Wetboek van economisch recht (artikel I.1, eer...",Onontvankelijkheid van de prejudiciële vraag,"[Economisch recht, Insolventie van onderneming...",www.const-court.be/public/n/2025/2025-060n.pdf
2,59/2025,[8285],03/04/2025,Beroep tot vernietiging,Ordonnantie van het Brusselse Hoofdstedelijke ...,Vernietiging,"[Fiscaal recht, Registratierechten, Brusselse ...",www.const-court.be/public/n/2025/2025-059n.pdf
3,58/2025,[8266],03/04/2025,Beroep tot vernietiging,Programmawet van 22 december 2023 (artikel 114),Verwerping van het beroep,"[Sociaal recht, Sociale zekerheid, Ziekte- en ...",www.const-court.be/public/n/2025/2025-058n.pdf
4,57/2025,"[8224, 8223]",03/04/2025,Prejudiciële vragen,Wet van 17 april 1878 « houdende de voorafgaan...,Geen schending (artikel 26 van de wet van 17 a...,"[Sociale zekerheid, Ziekte- en invaliditeitsve...",www.const-court.be/public/n/2025/2025-057n.pdf
...,...,...,...,...,...,...,...,...
56,5/2025,"[8157, 8156]",16/01/2025,Beroepen tot vernietiging,Decreet van het Vlaamse Gewest van 23 juni 202...,"- Vernietiging (artikel 5, eerste lid, 1°, van...","[Bestuursrecht, Grond- en pandenbeleid, Wonen ...",www.const-court.be/public/n/2025/2025-005n.pdf
57,4/2025,[8154],16/01/2025,Prejudiciële vraag,Wetboek van de inkomstenbelastingen 1992 (arti...,"Geen schending (artikel 269, § 2, eerste lid, ...","[Fiscaal recht, Inkomstenbelastingen, Personen...",www.const-court.be/public/n/2025/2025-004n.pdf
58,3/2025,[8148],16/01/2025,Beroep tot vernietiging,Ordonnantie van het Brusselse Hoofdstedelijke ...,- Vernietiging- Handhaving van de gevolgen van...,"[Ruimtelijke ordening, Brussels Hoofdstedelijk...",www.const-court.be/public/n/2025/2025-003n.pdf
59,2/2025,[8144],09/01/2025,Prejudiciële vraag,Brusselse Huisvestingscode (artikel 10),"Schending (artikel 10, § 3, van de Brusselse H...","[Huisvesting, Brussels Hoofdstedelijk Gewest, ...",www.const-court.be/public/n/2025/2025-002n.pdf


Subsequently, we write the dataframe to a pickle file, indicating temporal scope. When exporting to simpler format, such as \*.csv, the structure of some of the elements of the dataframe are lost (e.g. lists) (see https://predictivehacks.com/how-to-save-read-a-pandas-dataframe-containing-lists-and-dictionaries/) 

In [16]:
# # Create file_name for dataframe
# file_name = "CC_" + str(scope_years[0]) + "-" + str(scope_years[-1]) + ".csv"

# # Write away dataframe to excel for easier visual inspection at data_path location
# # (see https://www.geeksforgeeks.org/how-to-export-pandas-dataframe-to-a-csv-file/)
# data.to_csv(path_or_buf = download_folder + "\\" + file_name,
#             sep = "*", # Choose seperator that is least likely to appear inside strings of dataset
#             index = True,
#             encoding = "utf-16", # "utf-8" yielded difficulty with trema (ë, ï, ...)
#             quotechar = '"'
#            ) 

In [17]:
# Create file_name for dataframe
file_name = "CC_" + str(scope_years[0]) + "-" + str(scope_years[-1]) + ".pkl"

# Write away dataframe for easier visual inspection at data_path location
# see https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.to_pickle.html
# data.to_pickle(path = download_folder + "\\" + file_name) # In Windows, use "\\" instead of "/" in path
data.to_pickle(path = download_folder + "/" + file_name) # In Linux, use "/" in path

# Test chunks

In [18]:
# # %%time

# for year in scope_years: 
#     # Obtain BeautifulSoup object for each year in the range
#     bs = get_BeautifulSoup(str(year))
                           
#     # Obtain sections of html page that relate to each specific case
#     all_cases = bs.find_all("div",  {"class": "judgement-card mx-auto my-4 v-card v-sheet theme--light"})

#     # For each case, obtain the relevant information
#     for case in all_cases:
#         # Obtain all information that resides in top row of each judgement card (i.e. date and type of procedure)
#         for information in case.find_all("div", {"class":  "top-infos"}):
#             date, type_procedure = information.find_all("p")
#             cases_dates.append(date.get_text())
#             cases_type_procedure.append(type_procedure.get_text())

#         # Obtain case number and link. This is stored in first 'h3' heading , witin 'a'.
#         # attr. is used to extract the specific hyperlink itself as stored in 'href'
#         for information in case.find_all("h3"): 
#             # Obtain hyperlink
#             hyperlink = information.a.attrs["href"]
#             # Append hyperlink (+ add main website to hyperlink since this is always left out)
#             cases_hyperlink.append("www.const-court.be" + hyperlink)

#             # Obtain case number. There does not seem to be a specific tag for this.
#             # But the case number is always preceded with an empty html comment: "<!-- -->" (https://htmlcheatsheet.com/).
#             # Hence we use this to identify the case numbers within the string representation of information
#             location_comment = str(information).find("<!-- -->")
#             # select all text following this empty comment (this is the last element in the html string)
#             casenumber = str(information)[location_comment:]
#             # Strip resulting string of all trailing elements
#             casenumber = casenumber.replace("<!-- -->\n", "").replace("</h3>", "").replace("\n", "")
#             casenumber = casenumber.strip(" ")
#             cases_casenumber.append(casenumber) # append actual case number        


#         # Obtain keywords, which are stored in tag 'div' and class "v-banner__text"   
#         for information in case.find_all("div", {"class":  "v-banner__text"}):
#             keywords_str = information.get_text() # obtain text
#             keywords_list =  keywords_str.split(" - ") # split string into different keywords
#             # Strip each keyword of newlines and spaces
#             keywords_list = [keyword.replace("\n", "").strip(" ") for keyword in keywords_list] 
#             cases_keywords.append(keywords_list) # append to relevant list

#         # Obtain controlled_norm, outcome, rolenumber which are in that order stored in 'span' 
#         information = case.find_all("span")
#         controlled_norm, outcome, rolenumber = information   
#         # Append text of those elements (using '.get_text') to main lists, stripped if necessary
#         cases_controlled_norm.append(controlled_norm.get_text())
#         cases_outcome.append(outcome.get_text())
#         # Strip rolenumbers, split into list if more than one, and append
#         rolenumber = rolenumber.get_text().replace("\n", "").replace("Rolnummer: ", "").strip(" ")
#         rolenumber = rolenumber.split(" - ")
#         cases_rolenumber.append(rolenumber)

In [19]:
# import os
# os.getcwd()

In [20]:
# for case in all_cases:

    
# #     # Obtain controlled norms, which are stored in tag 'span' and class "subtitle my-2"   
# #     for information in case.find_all("span", {'class':  "subtitle my-2"}):
# #         cases_controlled_norm.append(information.get_text())

#     information = case.find_all("span")
#     controlled_norm, outcome, rolenumber = information
    
# #     for i in information:
# #         print(i)
# #         print('....')
# #         bestreden_norm, outcome, rolenumber = i
    
#     cases_controlled_norm.append(controlled_norm)
#     cases_outcome.append(outcome)
#     cases_rolenumber.append(rolenumber)
# bestreden_norm.get_text()
# bestreden_norm
# # info

In [21]:
# information.get_text()

In [22]:
# # Assign url with webpage containing judgements and corresponding keywords
# cases_url = "https://www.const-court.be/nl/judgments?year=2023"

# # Set headers to obtain language version NL 
# # see https://stackoverflow.com/questions/30561260/python-change-accept-language-using-requests + https://www.dataquest.io/blog/web-scraping-beautifulsoup/
# headers_nl = {"Accept-Language": "nl"} 

# # Obtain html code of CoC website for NL language version
# r = requests.get(cases_url, headers=headers_nl)

# # # Print html code
# # print(r.text)

In [23]:
# # Create BeautifulSoup object
# bs = BeautifulSoup(r.text, 'html.parser')

Subsequently, need to extract the relevant information about the cases from the webpage. For this, we need to inspect the htlm  code of the browser page (https://opensource.com/article/21/9/web-scraping-python-beautiful-soup).

In [24]:
# # Create BeautifulSoup object of page with results for specific year
# # for index_page, hyperlink_page in enumerate(hyperlinks_pages):  
# # html_page = urlopen(cases_hyperlink)
# data = requests.get(cases_hyperlink)
# bs = BeautifulSoup(data.text, 'html.parser')

# # Obtain relevant information for each case 
# hyperlinks_results = [] # link to download case
# case_numbers = []
# keywords = []

# case_section = bs_page.find_All("div", {"class": "judgement-card mx-auto my-4 v-card v-sheet theme--light"})

# # for result in bs_page.findAll('h2', {"class": "result"}):


# # Find all tags of h3 that have class "result"
# # For those you access child tag, i.e. "a". 
# # And for that you obtain the attribute "href", i.e. the hyperlink
# # for result in bs_page.findAll('h3', {"class": "result"}):
# for case in case_section:  
#     case_number = case.select(".v-banner__text")[0].get_text()
#     case_numbers.append(case_number)
#      # print(result)
        
# case_number
# case_numbers
# print(case_section.get_text())

In [25]:
# # Create BeautifulSoup object of page with results for specific year
# # for index_page, hyperlink_page in enumerate(hyperlinks_pages):  
# html_page = urlopen(cases_hyperlink)
# bs_page = BeautifulSoup(html_page.read(), 'html.parser')

# # Obtain relevant information for each case 
# hyperlinks_results = [] # link to download case
# case_number = []
# keywords = []


# # Find all tags of h3 that have class "result"
# # For those you access child tag, i.e. "a". 
# # And for that you obtain the attribute "href", i.e. the hyperlink
# # for result in bs_page.findAll('h3', {"class": "result"}):
# for result in bs_page.findAll('h3', {"class": "v-banner__text"}):  
#     print(result.a.attrs['href'])
#     # # recombine obtained link (i.e. result.a.attrs['href']) with main website and leave out ".." that occured in obtained hyperlinks
#     hyperlinks_results.append("http://www.raadvst-consetat.be" + result.a.attrs['href'][2:])
#     # get metadata of processed judgments
#     text.append(result.get_text())
# # hyperlinks_results   

# # For each of the obtained hyperlinks_results, download them to download_folder
# for index_result, hyperlink in enumerate(hyperlinks_results):
#     # # recombine obtained link with main website and leave out ".." that occured in obtained hyperlinks
#     # recombined = "http://www.raadvst-consetat.be" + hyperlink[2:] 
#     wget.download(hyperlink, download_folder)
#     print("{:>5} of {:>5} judgments downloaded".format(index_result + index_page * 10 + 1, count))

# # # progress
# # print("{:>2} of {:>3} pages downloaded".format(index_page + 1, len(hyperlinks_pages)))

In [26]:
# cases_list = str(bs_main).split('<div class="judgement-card mx-auto my-4 v-card v-sheet theme--light"')

In [27]:
# # cases_list[0] # This is the stuff written before the actual cases
# cases_list_clean = cases_list[1:]
# len(cases_list_clean)
# len(cases_list)
# cases_list_clean[0]

In [28]:
# print(bs_main.text)

In [29]:
# bs_main

In [30]:
# for i in case.find_all("span"):
#     print (i)
# # len(case.find_all("span"))

In [31]:
# information.attrs["subtitle my-2"]

In [32]:
# location_comment = str(information).find("<!-- -->")
# rolenumber = str(information)[location_comment:]
# rolenumber = rolenumber.replace("<!-- -->\n", "").replace("</h3>", "").replace("\n", "")
# rolenumber.strip()

In [33]:
# # cases_topinfos
# str(information).find("<!-- -->")
# # cases_hyperlink

In [34]:
# rolenumber

In [35]:
# bs_main.find('lang="fr-BE"')

In [36]:
# cases_keywords = bs.find_all('div', text = "v-banner__text" )
# cases_keywords
# print(cases_keywords)

In [37]:
# print(bs.get_text())

In [38]:
# # test chunk
# all_cases = bs.find_all("div",  {'class':  "judgement-card mx-auto my-4 v-card v-sheet theme--light"}, {'class':  "top-infos"})

# all_cases[0]
# len(all_cases) #14
# all_cases[0]
# all_cases2[0] == all_cases[0]

# #         element = test.find(class: "href")
            
# #         print(len(test[0]), test[0])
# #         print(information)
    
        
        
# #         a = information.p[0].get_text()
# #         print(a)
        
        
# #         date = information.find("p")
# #         cases_dates
# # ('h2', {"class": "result"})

# # cases_dates
# # cases_type_procedure
# #         test = information.find_all("a", {"class": "href"})

In [39]:
# ###############################################################################
# ## How to extract data from website RvS?
# ###############################################################################

# # https://likegeeks.com/downloading-files-using-python/



# ###############################################################################
# ###############################################################################
# #################### Extract pdf from single inputted link ####################
# ###############################################################################
# ###############################################################################


# ################ wget #########################################################
# # pip install wget

# import wget
# download_folder = "C:/Users/owner/Documents/Documenten/Data science/MaStat/Masterproef/data/RvS_judgments"
# url = "http://www.raadvst-consetat.be/Arresten/249000/100/249102.pdf#xml=http://www.raadvst-consetat.be/apps/dtsearch/getpdf.asp?DocId=41749&Index=c%3a%5csoftware%5cdtsearch%5cindex%5carrets%5fnl%5c&HitCount=7&hits=50+51+e4+e5+ff3+ff4+101b+&0850920221910"
# wget.download(url, download_folder)

# # --> is optie om documenten te downloaden eens je URL hebt van specifieke arrest


# ################ requests #####################################################
# import requests
# url = 'https://readthedocs.org/projects/python-guide/downloads/pdf/latest/'
# myfile = requests.get(url, allow_redirects=True)
# open('c:/users/LikeGeeks/documents/hello.pdf', 'wb').write(myfile.content)





# #http://www.raadvst-consetat.be/index.asp?page=caselaw_adv&lang=nl&booleanConditions=%28dat_arr+contains+%2820160101%7E%7E20200101%29%29+AND+%28num_rol+contains%28*XII*%29%29&qu=&method=and&index=arr&s_lang=nl&choixdate=E&jour_start=01&mois_start=01&annee_start=2016&jour_end=01&mois_end=01&annee_end=2020&num_arr_start=&num_arr_stop=&num_rol=*XII*&part=&dictum=
# start_date = "20180101" # format: yyyymmdd
# end_date = "20200101" # format: yyyymmdd

# url = "http://www.raadvst-consetat.be/index.asp?page=caselaw_adv&lang=nl&booleanConditions=%28dat_arr+contains+%28" + \
#         start_date + "%7E%7E" + end_date  + \
#         "%29%29+AND+%28num_rol+contains%28*XII*%29%29&qu=&method=and&index=arr&s_lang=nl&choixdate=E&jour_start=01&mois_start=01&annee_start=2016&jour_end=01&mois_end=01&annee_end=2020&num_arr_start=&num_arr_stop=&num_rol=*XII*&part=&dictum="

# print(url)


# """
# url = ''.join(("http://www.raadvst-consetat.be/index.asp?page=caselaw_adv&lang=nl&booleanConditions=%28dat_arr+contains+%28",
#                start_date,
#                "%7E%7E",
#                end_date,
#                "%29%29+AND+%28num_rol+contains%28*XII*%29%29&qu=&method=and&index=arr&s_lang=nl&choixdate=E&jour_start=01&mois_start=01&annee_start=2016&jour_end=01&mois_end=01&annee_end=2020&num_arr_start=&num_arr_stop=&num_rol=*XII*&part=&dictum="))
# print(url)
# url2 = "http://www.raadvst-consetat.be/index.asp?page=caselaw_adv&lang=nl&booleanConditions=%28dat_arr+contains+%28" + \
#         start_date + "%7E%7E" + end_date  + \
#         "%29%29+AND+%28num_rol+contains%28*XII*%29%29&qu=&method=and&index=arr&s_lang=nl&choixdate=E&jour_start=01&mois_start=01&annee_start=2016&jour_end=01&mois_end=01&annee_end=2020&num_arr_start=&num_arr_stop=&num_rol=*XII*&part=&dictum="
# math_result = 1 + 2 + 3 + 4 + \
#               5 + 6 + 7 + 8 + \
#               9 + 10
 
# print(message)
# print(math_result)
# url3 = ("http://www.raadvst-consetat.be/index.asp?page=caselaw_adv&lang=nl&booleanConditions=%28dat_arr+contains+%28" + 
#         start_date + "%7E%7E" + end_date  + 
#         "%29%29+AND+%28num_rol+contains%28*XII*%29%29&qu=&method=and&index=arr&s_lang=nl&choixdate=E&jour_start=01&mois_start=01&annee_start=2016&jour_end=01&mois_end=01&annee_end=2020&num_arr_start=&num_arr_stop=&num_rol=*XII*&part=&dictum="
#         )
# """

# ###############################################################################
# ###############################################################################
# ###### Extract all search results from single page of results #################
# ###############################################################################
# ###############################################################################


# # Download multiple files from website with python
# # https://stackoverflow.com/questions/31251293/how-to-download-multiple-files-and-images-from-a-website-using-python 
# import urllib

# website = urllib.request.urlopen(url).read() # modification from original code because of python 3x, see https://stackoverflow.com/questions/39975367/attributeerror-module-urllib-has-no-attribute-urlopen
# root = ET.fromstring(website)
# list = root.findall('table')
# hrefs = list.findall('a')
# for a in hrefs:
#   download(a)

# ###############################################
# ##############################################
# # Beautiful soup
# ###############################################
# ##############################################

# from urllib.request import urlopen
# from bs4 import BeautifulSoup
# html = urlopen(url)
# bs = BeautifulSoup(html.read(), 'html.parser')
# print(bs.h1)


# """
# nameList = bs.findAll('h2', {'class':'result'})
# nameList
# for name in nameList:
#     print(name.get_text())
# for name in nameList:
#     print(name.get_item())
    
    
# for name in bs.findAll('a', {'class':'result'}):
#     if 'href' in name.attrs:
#         print(name.attrs['href'])
# for name in bs.findAll('a'):
#     print(name.attrs)
    
# for item in bs.findAll('div', {"class": "item"}):
#     print(item.attrs)
    
# for result in bs.findAll('h2', {"class": "result"}):
#     print(result.attrs['href'])
    
# for a in bs.findAll('a'):
#     if 'href' in a.attrs:
#         print(a["href"])
        
# for result in bs.findAll('h2', {"class": "result"}):
#     for hyperlink in bs.findAll('a', {"class": "href"}):
#         print(hyperlink.attrs)
        
# hyperlinks = []               
# for result in bs.findAll('h2', {"class": "result"}):
#     hyperlinks.append(result.a['href'])
# """
# hyperlinks = []
# # Find all tags of h2 that have class "result"
# # For those you access child tag, i.e. "a". 
# # And for that you obtain the attribute "href", i.e. the hyperlink
# for result in bs.findAll('h2', {"class": "result"}):
#     #print(result.a.attrs['href'])
#     hyperlinks.append(result.a.attrs['href'])
# hyperlinks    
    

# download_folder = "C:/Users/owner/Documents/Documenten/Data science/MaStat/Masterproef/data/RvS_judgments"
# for hyperlink in hyperlinks:
#     recombined = "http://www.raadvst-consetat.be" + hyperlink[2:]
#     wget.download(recombined, download_folder)
    
# hyperlink[2:]


# ###############################################################################
# ###############################################################################
# ########### Extract results from multiple pages of results ####################
# ###############################################################################
# ###############################################################################

# test_url = "http://www.raadvst-consetat.be/index.asp?page=caselaw_adv&lang=nl&booleanConditions=%28dat_arr+contains+%2820160101%7E%7E20220101%29%29&qu=%22niels+tack%22&method=and&index=arr&s_lang=nl&choixdate=E&jour_start=01&mois_start=01&annee_start=2016&jour_end=01&mois_end=01&annee_end=2022&num_arr_start=&num_arr_stop=&num_rol=&part=&dictum="

# # Create BS object
# html_main = urlopen(test_url)
# bs_main = BeautifulSoup(html_main.read(), 'html.parser')


# # retrieve list of specific links to pages with results
# # no need to retrieve all times mentioned: those links appear multiple time on page, so 1e suffices (so only use bs.find instead of bs.findAll)
# hyperlinks_pages = []
# for page.a in bs_main.find('div', {'class': "searchresults_pages"}):
#     # print(page)
#     # append hyperlinks_pages with string of main website + retrieved hyperlink
#     hyperlinks_pages.append("http://www.raadvst-consetat.be" + '/' + page.a.attrs['href'])
    
# # hyperlinks_pages

# # Create new BeautifulSoup object, but now based on links of specific pages
# for index_page, hyperlink_page in enumerate(hyperlinks_pages):  
#     html_page = urlopen(hyperlink_page)
#     bs_page = BeautifulSoup(html_page.read(), 'html.parser')
      
#     # Obtain links to specific results of specific page with results
#     hyperlinks_results = []

#     # Find all tags of h2 that have class "result"
#     # For those you access child tag, i.e. "a". 
#     # And for that you obtain the attribute "href", i.e. the hyperlink
#     for result in bs_page.findAll('h2', {"class": "result"}):
#         #print(result.a.attrs['href'])
#         # # recombine obtained link (i.e. result.a.attrs['href']) with main website and leave out ".." that occured in obtained hyperlinks
#         hyperlinks_results.append("http://www.raadvst-consetat.be" + result.a.attrs['href'][2:])
#     # hyperlinks_results    
        
#     # For each of the obtained hyperlinks_results, download them to download_folder
#     for index_result, hyperlink in enumerate(hyperlinks_results):
#         # # recombine obtained link with main website and leave out ".." that occured in obtained hyperlinks
#         # recombined = "http://www.raadvst-consetat.be" + hyperlink[2:] 
#         wget.download(hyperlink, download_folder)
#         print("{:>5} of {:>5} judgments downloaded".format(index_result + 1, len(hyperlinks_results)))
        
#     # progress
#     print("{:>2} of {:>3} pages downloaded".format(index_page + 1, len(hyperlinks_pages)))

# #######################
# hyperlinks_pages = []
# # retrieve list specific links to pages with results
# # no need to retrieve all times mentioned: those links appear multiple time on page, so 1e suffices
# for page.a in bs.find('div', {'class': "searchresults_pages"}):
#     # print(page)
#     hyperlinks_pages.append(page.a.attrs['href'])
# ########################



# ###############################################################################
# ###############################################################################
# ###### Register metadata when downloading + count of total docs ###############
# ###############################################################################
# ###############################################################################



# import wget
# from urllib.request import urlopen
# from bs4 import BeautifulSoup

# download_folder = "C:/Users/owner/Documents/Documenten/Data science/MaStat/Masterproef/data/RvS_judgments"
# test_url = "http://www.raadvst-consetat.be/index.asp?page=caselaw_adv&lang=nl&booleanConditions=%28dat_arr+contains+%2820160101%7E%7E20220101%29%29&qu=%22niels+tack%22&method=and&index=arr&s_lang=nl&choixdate=E&jour_start=01&mois_start=01&annee_start=2016&jour_end=01&mois_end=01&annee_end=2022&num_arr_start=&num_arr_stop=&num_rol=&part=&dictum="

# # Create BS object for general search
# html_main = urlopen(test_url)
# bs_main = BeautifulSoup(html_main.read(), 'html.parser')

# # create empty list to gather metadata of judgments
# text = []


# # Get total of found documents
# count_text = bs_main.find('div', {'class': "infofoundcount"})
# count_text = count_text.get_text()
# if count_text.startswith('Er werden'): # Dutch inferface
#     count = count_text.split()[2] # obtain third element, i.e. actual count
# elif count_text.endswith('documents ont été trouvés'):# French interface
#     count = count_text.split()[0]
# else:   
#     print("Error: Language interface in Dutch or French?")


# # retrieve list of specific links to pages with results
# # no need to retrieve all times mentioned: those links appear multiple time on page, so 1e suffices (so only use bs.find instead of bs.findAll)
# hyperlinks_pages = []
# for page in bs_main.find('div', {'class': "searchresults_pages"}):
#     # print(page)
#     # append hyperlinks_pages with string of main website + retrieved hyperlink
#     hyperlinks_pages.append("http://www.raadvst-consetat.be" + '/' + page.attrs['href'])
    
# # hyperlinks_pages



# # Create new BeautifulSoup object, but now based on links of specific pages (so for each page with results)
# for index_page, hyperlink_page in enumerate(hyperlinks_pages):  
#     html_page = urlopen(hyperlink_page)
#     bs_page = BeautifulSoup(html_page.read(), 'html.parser')
      
#     # Obtain links to specific results of specific page with results
#     hyperlinks_results = []

#     # Find all tags of h2 that have class "result"
#     # For those you access child tag, i.e. "a". 
#     # And for that you obtain the attribute "href", i.e. the hyperlink
#     for result in bs_page.findAll('h2', {"class": "result"}):
#         #print(result.a.attrs['href'])
#         # # recombine obtained link (i.e. result.a.attrs['href']) with main website and leave out ".." that occured in obtained hyperlinks
#         hyperlinks_results.append("http://www.raadvst-consetat.be" + result.a.attrs['href'][2:])
#         # get metadata of processed judgments
#         text.append(result.get_text())
#     # hyperlinks_results    
        
#     # For each of the obtained hyperlinks_results, download them to download_folder
#     for index_result, hyperlink in enumerate(hyperlinks_results):
#         # # recombine obtained link with main website and leave out ".." that occured in obtained hyperlinks
#         # recombined = "http://www.raadvst-consetat.be" + hyperlink[2:] 
#         wget.download(hyperlink, download_folder)
#         print("{:>5} of {:>5} judgments downloaded".format(index_result + 1, len(hyperlinks_results)))
        
#     # progress
#     print("{:>2} of {:>3} pages downloaded".format(index_page + 1, len(hyperlinks_pages)))

# references = []
# dates = []
# numbers = []
# for metadata in text:
#     if metadata.startswith("Arrest."): # Dutch judgment
#         nr_and_date, dock_nr = metadata.split(",")
#         # add fixed text for reference field + first 3 digits + dot + rest of nr_and_date
#         references.append("R.v.St. nr. " + nr_and_date[8:11] + "." + nr_and_date[11:])      
#         numbers.append(nr_and_date[8:11] + " " + nr_and_date[11:14])     
#         dates.append(nr_and_date[-10:])
#     elif metadata.starstwith("Arrêt."): # French judgment
#         pass
#     else: 
#         print("Error: No Dutch or French judgment?")
        
   
        
# # register metadata in panda
# import os
# import pandas as pd

# # Set working directory to folder with code
# path_code = "C:/Users/owner/Documents/Documenten/Data science/MaStat/Masterproef/data"
# os.chdir(path_code)

# # read in data (only sample at this point)
# train_data = pd.read_excel('sample.xls')
# # train_data.head()
# # train_data.shape
# # There are 12 entries and 7 variables

# ##############################################################################
# ######### Data exploration
# ##############################################################################

# # Inspect keywords: create set of all keywords
# distinct_keywords = train_data['Keyword(s)']
# for keywords in dis
# distinct_keywords_set = i.split(";") for i in train_data['Keyword(s)']]

In [40]:
# ###############################################################################
# ## How to extract data from website RvS?
# ###############################################################################

# # https://likegeeks.com/downloading-files-using-python/



# ###############################################################################
# ###############################################################################
# #################### Extract pdf from single inputted link ####################
# ###############################################################################
# ###############################################################################


# ################ wget #########################################################
# # pip install wget

# import wget
# download_folder = "C:/Users/owner/Documents/Documenten/Data science/MaStat/Masterproef/data/RvS_judgments"
# url = "http://www.raadvst-consetat.be/Arresten/249000/100/249102.pdf#xml=http://www.raadvst-consetat.be/apps/dtsearch/getpdf.asp?DocId=41749&Index=c%3a%5csoftware%5cdtsearch%5cindex%5carrets%5fnl%5c&HitCount=7&hits=50+51+e4+e5+ff3+ff4+101b+&0850920221910"
# wget.download(url, download_folder)

# # --> is optie om documenten te downloaden eens je URL hebt van specifieke arrest


# ################ requests #####################################################
# import requests
# url = 'https://readthedocs.org/projects/python-guide/downloads/pdf/latest/'
# myfile = requests.get(url, allow_redirects=True)
# open('c:/users/LikeGeeks/documents/hello.pdf', 'wb').write(myfile.content)





# #http://www.raadvst-consetat.be/index.asp?page=caselaw_adv&lang=nl&booleanConditions=%28dat_arr+contains+%2820160101%7E%7E20200101%29%29+AND+%28num_rol+contains%28*XII*%29%29&qu=&method=and&index=arr&s_lang=nl&choixdate=E&jour_start=01&mois_start=01&annee_start=2016&jour_end=01&mois_end=01&annee_end=2020&num_arr_start=&num_arr_stop=&num_rol=*XII*&part=&dictum=
# start_date = "20180101" # format: yyyymmdd
# end_date = "20200101" # format: yyyymmdd

# url = "http://www.raadvst-consetat.be/index.asp?page=caselaw_adv&lang=nl&booleanConditions=%28dat_arr+contains+%28" + \
#         start_date + "%7E%7E" + end_date  + \
#         "%29%29+AND+%28num_rol+contains%28*XII*%29%29&qu=&method=and&index=arr&s_lang=nl&choixdate=E&jour_start=01&mois_start=01&annee_start=2016&jour_end=01&mois_end=01&annee_end=2020&num_arr_start=&num_arr_stop=&num_rol=*XII*&part=&dictum="

# print(url)


# """
# url = ''.join(("http://www.raadvst-consetat.be/index.asp?page=caselaw_adv&lang=nl&booleanConditions=%28dat_arr+contains+%28",
#                start_date,
#                "%7E%7E",
#                end_date,
#                "%29%29+AND+%28num_rol+contains%28*XII*%29%29&qu=&method=and&index=arr&s_lang=nl&choixdate=E&jour_start=01&mois_start=01&annee_start=2016&jour_end=01&mois_end=01&annee_end=2020&num_arr_start=&num_arr_stop=&num_rol=*XII*&part=&dictum="))
# print(url)
# url2 = "http://www.raadvst-consetat.be/index.asp?page=caselaw_adv&lang=nl&booleanConditions=%28dat_arr+contains+%28" + \
#         start_date + "%7E%7E" + end_date  + \
#         "%29%29+AND+%28num_rol+contains%28*XII*%29%29&qu=&method=and&index=arr&s_lang=nl&choixdate=E&jour_start=01&mois_start=01&annee_start=2016&jour_end=01&mois_end=01&annee_end=2020&num_arr_start=&num_arr_stop=&num_rol=*XII*&part=&dictum="
# math_result = 1 + 2 + 3 + 4 + \
#               5 + 6 + 7 + 8 + \
#               9 + 10
 
# print(message)
# print(math_result)
# url3 = ("http://www.raadvst-consetat.be/index.asp?page=caselaw_adv&lang=nl&booleanConditions=%28dat_arr+contains+%28" + 
#         start_date + "%7E%7E" + end_date  + 
#         "%29%29+AND+%28num_rol+contains%28*XII*%29%29&qu=&method=and&index=arr&s_lang=nl&choixdate=E&jour_start=01&mois_start=01&annee_start=2016&jour_end=01&mois_end=01&annee_end=2020&num_arr_start=&num_arr_stop=&num_rol=*XII*&part=&dictum="
#         )
# """

# ###############################################################################
# ###############################################################################
# ###### Extract all search results from single page of results #################
# ###############################################################################
# ###############################################################################


# # Download multiple files from website with python
# # https://stackoverflow.com/questions/31251293/how-to-download-multiple-files-and-images-from-a-website-using-python 
# import urllib

# website = urllib.request.urlopen(url).read() # modification from original code because of python 3x, see https://stackoverflow.com/questions/39975367/attributeerror-module-urllib-has-no-attribute-urlopen
# root = ET.fromstring(website)
# list = root.findall('table')
# hrefs = list.findall('a')
# for a in hrefs:
#   download(a)

# ###############################################
# ##############################################
# # Beautiful soup
# ###############################################
# ##############################################

# from urllib.request import urlopen
# from bs4 import BeautifulSoup
# html = urlopen(url)
# bs = BeautifulSoup(html.read(), 'html.parser')
# print(bs.h1)


# """
# nameList = bs.findAll('h2', {'class':'result'})
# nameList
# for name in nameList:
#     print(name.get_text())
# for name in nameList:
#     print(name.get_item())
    
    
# for name in bs.findAll('a', {'class':'result'}):
#     if 'href' in name.attrs:
#         print(name.attrs['href'])
# for name in bs.findAll('a'):
#     print(name.attrs)
    
# for item in bs.findAll('div', {"class": "item"}):
#     print(item.attrs)
    
# for result in bs.findAll('h2', {"class": "result"}):
#     print(result.attrs['href'])
    
# for a in bs.findAll('a'):
#     if 'href' in a.attrs:
#         print(a["href"])
        
# for result in bs.findAll('h2', {"class": "result"}):
#     for hyperlink in bs.findAll('a', {"class": "href"}):
#         print(hyperlink.attrs)
        
# hyperlinks = []               
# for result in bs.findAll('h2', {"class": "result"}):
#     hyperlinks.append(result.a['href'])
# """
# hyperlinks = []
# # Find all tags of h2 that have class "result"
# # For those you access child tag, i.e. "a". 
# # And for that you obtain the attribute "href", i.e. the hyperlink
# for result in bs.findAll('h2', {"class": "result"}):
#     #print(result.a.attrs['href'])
#     hyperlinks.append(result.a.attrs['href'])
# hyperlinks    
    

# download_folder = "C:/Users/owner/Documents/Documenten/Data science/MaStat/Masterproef/data/RvS_judgments"
# for hyperlink in hyperlinks:
#     recombined = "http://www.raadvst-consetat.be" + hyperlink[2:]
#     wget.download(recombined, download_folder)
    
# hyperlink[2:]


# ###############################################################################
# ###############################################################################
# ########### Extract results from multiple pages of results ####################
# ###############################################################################
# ###############################################################################

# test_url = "http://www.raadvst-consetat.be/index.asp?page=caselaw_adv&lang=nl&booleanConditions=%28dat_arr+contains+%2820160101%7E%7E20220101%29%29&qu=%22niels+tack%22&method=and&index=arr&s_lang=nl&choixdate=E&jour_start=01&mois_start=01&annee_start=2016&jour_end=01&mois_end=01&annee_end=2022&num_arr_start=&num_arr_stop=&num_rol=&part=&dictum="

# # Create BS object
# html_main = urlopen(test_url)
# bs_main = BeautifulSoup(html_main.read(), 'html.parser')


# # retrieve list of specific links to pages with results
# # no need to retrieve all times mentioned: those links appear multiple time on page, so 1e suffices (so only use bs.find instead of bs.findAll)
# hyperlinks_pages = []
# for page.a in bs_main.find('div', {'class': "searchresults_pages"}):
#     # print(page)
#     # append hyperlinks_pages with string of main website + retrieved hyperlink
#     hyperlinks_pages.append("http://www.raadvst-consetat.be" + '/' + page.a.attrs['href'])
    
# # hyperlinks_pages

# # Create new BeautifulSoup object, but now based on links of specific pages
# for index_page, hyperlink_page in enumerate(hyperlinks_pages):  
#     html_page = urlopen(hyperlink_page)
#     bs_page = BeautifulSoup(html_page.read(), 'html.parser')
      
#     # Obtain links to specific results of specific page with results
#     hyperlinks_results = []

#     # Find all tags of h2 that have class "result"
#     # For those you access child tag, i.e. "a". 
#     # And for that you obtain the attribute "href", i.e. the hyperlink
#     for result in bs_page.findAll('h2', {"class": "result"}):
#         #print(result.a.attrs['href'])
#         # # recombine obtained link (i.e. result.a.attrs['href']) with main website and leave out ".." that occured in obtained hyperlinks
#         hyperlinks_results.append("http://www.raadvst-consetat.be" + result.a.attrs['href'][2:])
#     # hyperlinks_results    
        
#     # For each of the obtained hyperlinks_results, download them to download_folder
#     for index_result, hyperlink in enumerate(hyperlinks_results):
#         # # recombine obtained link with main website and leave out ".." that occured in obtained hyperlinks
#         # recombined = "http://www.raadvst-consetat.be" + hyperlink[2:] 
#         wget.download(hyperlink, download_folder)
#         print("{:>5} of {:>5} judgments downloaded".format(index_result + 1, len(hyperlinks_results)))
        
#     # progress
#     print("{:>2} of {:>3} pages downloaded".format(index_page + 1, len(hyperlinks_pages)))

# #######################
# hyperlinks_pages = []
# # retrieve list specific links to pages with results
# # no need to retrieve all times mentioned: those links appear multiple time on page, so 1e suffices
# for page.a in bs.find('div', {'class': "searchresults_pages"}):
#     # print(page)
#     hyperlinks_pages.append(page.a.attrs['href'])
# ########################



# ###############################################################################
# ###############################################################################
# ###### Register metadata when downloading + count of total docs ###############
# ###############################################################################
# ###############################################################################



# import wget
# from urllib.request import urlopen
# from bs4 import BeautifulSoup

# download_folder = "C:/Users/owner/Documents/Documenten/Data science/MaStat/Masterproef/data/RvS_judgments"
# test_url = "http://www.raadvst-consetat.be/index.asp?page=caselaw_adv&lang=nl&booleanConditions=%28dat_arr+contains+%2820160101%7E%7E20220101%29%29&qu=%22niels+tack%22&method=and&index=arr&s_lang=nl&choixdate=E&jour_start=01&mois_start=01&annee_start=2016&jour_end=01&mois_end=01&annee_end=2022&num_arr_start=&num_arr_stop=&num_rol=&part=&dictum="

# # Create BS object for general search
# html_main = urlopen(test_url)
# bs_main = BeautifulSoup(html_main.read(), 'html.parser')

# # create empty list to gather metadata of judgments
# text = []


# # Get total of found documents
# count_text = bs_main.find('div', {'class': "infofoundcount"})
# count_text = count_text.get_text()
# if count_text.startswith('Er werden'): # Dutch inferface
#     count = count_text.split()[2] # obtain third element, i.e. actual count
# elif count_text.endswith('documents ont été trouvés'):# French interface
#     count = count_text.split()[0]
# else:   
#     print("Error: Language interface in Dutch or French?")


# # retrieve list of specific links to pages with results
# # no need to retrieve all times mentioned: those links appear multiple time on page, so 1e suffices (so only use bs.find instead of bs.findAll)
# hyperlinks_pages = []
# for page in bs_main.find('div', {'class': "searchresults_pages"}):
#     # print(page)
#     # append hyperlinks_pages with string of main website + retrieved hyperlink
#     hyperlinks_pages.append("http://www.raadvst-consetat.be" + '/' + page.attrs['href'])
    
# # hyperlinks_pages



# # Create new BeautifulSoup object, but now based on links of specific pages (so for each page with results)
# for index_page, hyperlink_page in enumerate(hyperlinks_pages):  
#     html_page = urlopen(hyperlink_page)
#     bs_page = BeautifulSoup(html_page.read(), 'html.parser')
      
#     # Obtain links to specific results of specific page with results
#     hyperlinks_results = []

#     # Find all tags of h2 that have class "result"
#     # For those you access child tag, i.e. "a". 
#     # And for that you obtain the attribute "href", i.e. the hyperlink
#     for result in bs_page.findAll('h2', {"class": "result"}):
#         #print(result.a.attrs['href'])
#         # # recombine obtained link (i.e. result.a.attrs['href']) with main website and leave out ".." that occured in obtained hyperlinks
#         hyperlinks_results.append("http://www.raadvst-consetat.be" + result.a.attrs['href'][2:])
#         # get metadata of processed judgments
#         text.append(result.get_text())
#     # hyperlinks_results    
        
#     # For each of the obtained hyperlinks_results, download them to download_folder
#     for index_result, hyperlink in enumerate(hyperlinks_results):
#         # # recombine obtained link with main website and leave out ".." that occured in obtained hyperlinks
#         # recombined = "http://www.raadvst-consetat.be" + hyperlink[2:] 
#         wget.download(hyperlink, download_folder)
#         print("{:>5} of {:>5} judgments downloaded".format(index_result + 1, len(hyperlinks_results)))
        
#     # progress
#     print("{:>2} of {:>3} pages downloaded".format(index_page + 1, len(hyperlinks_pages)))

# references = []
# dates = []
# numbers = []
# for metadata in text:
#     if metadata.startswith("Arrest."): # Dutch judgment
#         nr_and_date, dock_nr = metadata.split(",")
#         # add fixed text for reference field + first 3 digits + dot + rest of nr_and_date
#         references.append("R.v.St. nr. " + nr_and_date[8:11] + "." + nr_and_date[11:])      
#         numbers.append(nr_and_date[8:11] + " " + nr_and_date[11:14])     
#         dates.append(nr_and_date[-10:])
#     elif metadata.starstwith("Arrêt."): # French judgment
#         pass
#     else: 
#         print("Error: No Dutch or French judgment?")
        
   
        
# # register metadata in panda
# import os
# import pandas as pd

# # Set working directory to folder with code
# path_code = "C:/Users/owner/Documents/Documenten/Data science/MaStat/Masterproef/data"
# os.chdir(path_code)

# # read in data (only sample at this point)
# train_data = pd.read_excel('sample.xls')
# # train_data.head()
# # train_data.shape
# # There are 12 entries and 7 variables

# ##############################################################################
# ######### Data exploration
# ##############################################################################

# # Inspect keywords: create set of all keywords
# distinct_keywords = train_data['Keyword(s)']
# for keywords in dis
# distinct_keywords_set = i.split(";") for i in train_data['Keyword(s)']]

In [41]:
# # create empty list to gather metadata of judgments
# text = []

# # Get total of found documents
# count_text = bs_main.find('div', {'class': "infofoundcount"})
# count_text = count_text.get_text()
# if count_text.startswith('Er werden'): # Dutch inferface
#     count = count_text.split()[2] # obtain third element, i.e. actual count
# elif count_text.endswith('documents ont été trouvés'):# French interface
#     count = count_text.split()[0]
# else:   
#     print("Error: Language interface in Dutch or French?")


# # retrieve list of specific links to pages with results
# # no need to retrieve all times mentioned: those links appear multiple time on page, so 1e suffices (so only use bs.find instead of bs.findAll)
# hyperlinks_pages = []
# for page in bs_main.find('div', {'class': "searchresults_pages"}):
#     # print(page)
#     # append hyperlinks_pages with string of main website + retrieved hyperlink
#     hyperlinks_pages.append("http://www.raadvst-consetat.be" + '/' + page.attrs['href'])
    
# # hyperlinks_pages

# # Create new BeautifulSoup object, but now based on links of specific pages (so for each page with results)
# for index_page, hyperlink_page in enumerate(hyperlinks_pages):  
#     html_page = urlopen(hyperlink_page)
#     bs_page = BeautifulSoup(html_page.read(), 'html.parser')
      
#     # Obtain links to specific results of specific page with results
#     hyperlinks_results = []

#     # Find all tags of h2 that have class "result"
#     # For those you access child tag, i.e. "a". 
#     # And for that you obtain the attribute "href", i.e. the hyperlink
#     for result in bs_page.findAll('h2', {"class": "result"}):
#         #print(result.a.attrs['href'])
#         # # recombine obtained link (i.e. result.a.attrs['href']) with main website and leave out ".." that occured in obtained hyperlinks
#         hyperlinks_results.append("http://www.raadvst-consetat.be" + result.a.attrs['href'][2:])
#         # get metadata of processed judgments
#         text.append(result.get_text())
#     # hyperlinks_results    
        
#     # For each of the obtained hyperlinks_results, download them to download_folder
#     for index_result, hyperlink in enumerate(hyperlinks_results):
#         # # recombine obtained link with main website and leave out ".." that occured in obtained hyperlinks
#         # recombined = "http://www.raadvst-consetat.be" + hyperlink[2:] 
#         wget.download(hyperlink, download_folder)
#         print("{:>5} of {:>5} judgments downloaded".format(index_result + index_page * 10 + 1, count))
        
#     # # progress
#     # print("{:>2} of {:>3} pages downloaded".format(index_page + 1, len(hyperlinks_pages)))

Possible indicators of a new judgment
<div class="judgement-card
<div class="judgement-card mx-auto my-4 v-card v-sheet theme--light"
</div></div><div class="judgement-card mx-auto my-4 v-card v-sheet theme--light"
<div class="v-dialog__container">
</div></div></div></div> <div class="v-dialog__container">
<!-- -->
<!-- --></div></div><div class="judgement-card mx-auto my-4 v-card v-sheet theme--light"
</div></div><div class="judgement-card mx-auto my-4 v-card v-sheet theme--light"
<div></div></div></div> <div class="v-dialog__container"><!-- --></div></div><div class="judgement-card mx-auto my-4 v-card v-sheet theme--light"
<div class="judgement-card mx-auto my-4 v-card v-sheet theme--light"
                                                                    
                                                                    
* om te starten: <div class="judgement-card mx-auto my-4 v-card v-sheet theme--light" 
    (in meeste gevallen gaat dit vooraf met </div></div>, maar niet in eerste case) (dit levert wel één resultaat te veel op)
* om af te sluiten: </div></div></div></div> <div class="v-dialog__container">
* aanduiding begin trefwoorden: Mots-clés:   </p> <div class="v-banner subtitle my-2 v-sheet theme--light elevation-3 v-banner--is-mobile"><div class="v-banner__wrapper"><div class="v-banner__content"><div class="v-banner__text">
* aanduiding einde trefwoorden: </div></div></div></div> <div class="v-dialog__container">
                                                                                         
          
* aanduiding begin van relevante htlm code voor case informatie: <div id="alljudgmentcards"
* 

In [42]:
#    # Obtain all information that is retained in the 'span' tags, i.e. controlled norm, outcome, keywords and role number
#     for information in case.find_all("span"):
#         information.attrs["subtitle my-2"]

In [43]:
#     # Obtain controlled norms, which are stored in tag 'span' and class "subtitle my-2"   
#     for information in case.find_all("span", {'class':  "subtitle my-2"}):
#         cases_controlled_norm.append(information.get_text())

## Setting language

In [44]:
# # Create BS object for general search

# html = urlopen(url, headers = headers_nl)
# bs = BeautifulSoup(html.read(), 'html.parser')

In [45]:

# url = urlopen.get(cases_url, headers=headers_nl)
# url

In [46]:
# test = bs.body.find_all("nav")
# len(test)
# test[1]

In [47]:
#Navigate through structure of CoC website to access language setting
# bs.body.aside.div.section.select

In [48]:
#Navigate through structure of CoC website to access language setting
# bs.body.aside.div.section.find_all("option")

In [49]:
# bs.html.lang

In [50]:
# for i in bs.body.children:
#     print(i)

## Correcting data

--> issues with 
* entry 977 (Vlaamse decreet van 26 maart 2004)
* entry 573 (Programmawet van 30 december 2001 (art. 116, 117, 131 en 168, 13de en 15de streepje) Koninklijk besluit van 30 maart 2001 tot regeling van de rechtspositie van het personeel van de politiediensten (deel XII bekrachtigd bij artikel 131 van de programmawet van 30 december 2001 en artikel IV.I.7 bekrachtigd bij artikel 136 van de wet van 26 april 2002)Wet van 26 april 2002 houdende de essentiële elementen van het statuut van de personeelsleden van de politiediensten en houdende diverse andere bepalingen met betrekking tot de politiediensten)
* entry 1010: ['Gerechtelijk recht', 'Burgerlijke rechtspleging', 'Geschil betreffende de erelonen van advocaten', 'Beslissing van de raad van de Orde', 'Hoger beroep.']
* 1535 and 1536: seem to bes plit (rolenumber 4274-4199)


In [51]:
# # cases_rolenumber
# controlled_norm, outcome, rolenumber = information
# rolenumber
# rolenumber = rolenumber.get_text().replace("\n", "").replace("Rolnummer: ", "").strip(" ")
# rolenumber
# rolenumber = rolenumber.split(" - ")
# rolenumber

In [52]:
# cases_rolenumber[38]
# # .split(" - ")
# len(cases_rolenumber)
# len(cases_casenumber)
# len(cases_hyperlink)
# len(cases_casenumber)
# len(cases_dates)
# len(cases_type_procedure)   
# len(cases_controlled_norm)
# len(cases_keywords)
# len(cases_outcome)
# len(cases_rolenumber)

## Drop missing values for "keywords"

In [53]:
# data2 = data[:]
# data2

In [54]:
# keys = list(missing_values_keywords_df.columns.values)
# i1 = data.set_index(keys).index
# i2 = missing_values_keywords_df.set_index(keys).index
# data_no_missing = data[i1.isin(i2)]
# data_no_missing

In [55]:
# data3 = data.merge(missing_values_keywords_df,
#                   on = "keywords",
#                   how = "left",
#                   indicator = True)
# .query('_merge == "left_only"')
# .drop(columns='_merge')

    
# data3

In [56]:
# # data2 = data2[data2 =! missing_values_keywords_bool]
# # data2 = data2[data2[missing_values_keywords_bool]== False]
# data2 = pd.merge(data, missing_values_keywords_df, how = 'outer', on = ["keywords"])
# data2

In [57]:
# data[missing_values_keywords_bool]

In [58]:
# # Create boolean that indicates if a case has no keywords assigned
# missing_values_keywords = data[data["keywords"].str == "-"]
# missing_values_keywords

# # data[data["keywords"] == ["-"]].index
# # data[missing_values_keywords]

In [59]:
# data4 = data.drop(data[missing_values_keywords_bool].index)
# data4

In [60]:
# len(data[missing_values_keywords_bool])